## Match to OpenAlex

### What we do

All papers are already matched to OpenAlex:
- **Huang conferences (29)**: previously matched and saved in `huang_matched_openalex.csv` (890 rows)
- **ICWSM + JCDL (2)**: matched in notebook `01b` and saved in `icwsm_jcdl_awards_raw.csv` (35 rows in pilot window)

This notebook just aligns columns and merges the two sources into one unified `huang_matched_openalex.csv`.

In [ ]:
import pandas as pd

BASE = 'B:\\Semester 4 UU\\thesis-best-paper-trajectories\\data\\'

# ── Load existing matched data ───────────────────────────────────────────────────
huang = pd.read_csv(BASE + 'matched\\huang_matched_openalex.csv')
icwsm_jcdl = pd.read_csv(BASE + 'raw\\icwsm_jcdl_awards_raw.csv')

# keep only pilot window (2000-2018) for ICWSM/JCDL
icwsm_jcdl = icwsm_jcdl[icwsm_jcdl['year'].between(2000, 2018)].copy()

print(f'Huang rows:      {len(huang)}')
print(f'ICWSM/JCDL rows: {len(icwsm_jcdl)}')

# ── Align columns ────────────────────────────────────────────────────────────
# icwsm_jcdl uses 'openalex_title' — rename to match huang's 'oa_title'
icwsm_jcdl = icwsm_jcdl.rename(columns={'openalex_title': 'oa_title'})
icwsm_jcdl['match_route'] = 'pre-matched'

# keep only the shared columns
COLS = ['year', 'conference', 'paper_title', 'paper_url', 'authors',
        'openalex_id', 'doi', 'match_route', 'oa_title', 'authorships']

for col in COLS:
    if col not in huang.columns:       huang[col]       = None
    if col not in icwsm_jcdl.columns:  icwsm_jcdl[col]  = None

# ── Merge ───────────────────────────────────────────────────────────────────
out = pd.concat([huang[COLS], icwsm_jcdl[COLS]], ignore_index=True)

out.to_csv(BASE + 'matched\\huang_matched_openalex.csv', index=False)

matched = out['openalex_id'].notna().sum()
print(f"\n{'='*50}")
print(f"Total: {len(out)} | Matched: {matched} ({matched/len(out)*100:.1f}%)")
print(out.groupby('match_route')['openalex_id'].apply(lambda x: x.notna().sum()))


#### Check — flag low-confidence matches

In [ ]:
from rapidfuzz import fuzz
out['title_similarity'] = out.apply(
    lambda r: 100 if r['match_route'] in ('SS→DOI→OA', 'pre-matched')
    else fuzz.ratio(str(r['paper_title']).lower(), str(r['oa_title']).lower()),
    axis=1
)
out['low_confidence'] = out['title_similarity'] < 85
print(out['low_confidence'].sum(), 'flagged for manual review')


#### Manual Checkups

In [ ]:
unmatched = out[out['openalex_id'].isna()]
unmatched[['year','conference','paper_title']].to_csv(BASE + 'matched\\unmatched_manual.csv', index=False)
